# Ejercicio 7: visualización de la adenilato quinasa abierta y cerrada

Las dos formas (1AKE cerrada, 4AKE abierta) coloreadas por dominio, primero por separado y después superpuestas. Los cálculos del ejercicio se corren con `venv/bin/python main.py 7`; este notebook solo dibuja.

Correr desde la raíz del TP con `venv/bin/jupyter lab ex7_visualizacion.ipynb`.

In [1]:
import nglview as nv
from structures import load

ake = load('1AKE')    # adenilato quinasa cerrada, con Ap5A
ake_open = load('4AKE')    # adenilato quinasa abierta, sin ligando

/Users/juliazack/Documents/fcen/bioinfo/bioinfo_tp5/venv/lib/python3.12/site-packages/nglview/__init__.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## Forma cerrada (1AKE)

CORE en gris, NMP (30–59) en azul, LID (122–159) en rojo (límites de dominio tomados de la literatura sobre la enzima, no del archivo), Ap5A como bastones: en la forma cerrada la tapa se pliega sobre el inhibidor.

In [2]:
v = nv.show_biopython(ake)
v.clear_representations()
v.add_representation('cartoon', selection=':A', color='grey')
v.add_representation('cartoon', selection=':A and 30-59', color='blue')
v.add_representation('cartoon', selection=':A and 122-159', color='red')
v.add_representation('licorice', selection=':A and AP5', color='yellow')
v

NGLWidget()

## Forma abierta (4AKE)

Mismos colores. No hay ligando: la estructura se obtuvo sin moléculas unidas. El NMP (azul) y el LID (rojo) quedan separados del CORE, dejando abierto el hueco donde en la forma cerrada está el Ap5A.

In [3]:
v = nv.show_biopython(ake_open)
v.clear_representations()
v.add_representation('cartoon', selection=':A', color='grey')
v.add_representation('cartoon', selection=':A and 30-59', color='blue')
v.add_representation('cartoon', selection=':A and 122-159', color='red')
v

NGLWidget()

## Las dos formas superpuestas por el CORE

Se mueve una copia de la abierta (4AKE) para que su CORE quede lo más encimado posible sobre el de la cerrada, usando solo los carbonos alfa del CORE. Así el CORE coincide y lo que queda desplazado es el movimiento de los dominios móviles. El RMSD de cada dominio después de superponer mide ese desplazamiento: chico en el CORE (es lo que se alineó), grande en el NMP y el LID.

In [4]:
from exercises.ex7.ex7 import DOMAINS
from geometry import rmsd, superimpose

closed_fit = load('1AKE')   # referencia, no se mueve
open_fit = load('4AKE')     # copia aparte: la de arriba queda como estaba

def domain_cas(structure, domain):
    chain = structure[0]['A']
    return [chain[n]['CA'] for n in DOMAINS[domain]]

sup = superimpose(domain_cas(closed_fit, 'CORE'), domain_cas(open_fit, 'CORE'), open_fit)
for domain in ('CORE', 'NMP', 'LID'):
    print(f'RMSD {domain:<4} después de superponer por el CORE: '
          f'{rmsd(domain_cas(closed_fit, domain), domain_cas(open_fit, domain)):5.2f} Å')

RMSD CORE después de superponer por el CORE:  1.98 Å
RMSD NMP  después de superponer por el CORE: 11.23 Å
RMSD LID  después de superponer por el CORE: 15.58 Å


Cerrada en colores fuertes (gris, azul, rojo) y abierta en los mismos colores en versión clara: mismo tono = mismo dominio, oscuro = cerrada, claro = abierta. El Ap5A (amarillo) es de la cerrada.

In [5]:
v = nv.NGLWidget()
v.add_component(nv.BiopythonStructure(closed_fit))   # componente 0
v.add_component(nv.BiopythonStructure(open_fit))     # componente 1
for c in (0, 1):
    v.clear_representations(component=c)

for c, (core, nmp, lid) in enumerate([('grey', 'blue', 'red'), ('lightgrey', 'lightblue', 'pink')]):
    v.add_representation('cartoon', selection=':A', color=core, component=c)
    v.add_representation('cartoon', selection=':A and 30-59', color=nmp, component=c)
    v.add_representation('cartoon', selection=':A and 122-159', color=lid, component=c)
v.add_representation('licorice', selection=':A and AP5', color='yellow', component=0)
v

NGLWidget()